#Between Fall 2021 and Fall 2025 (time window), how did the freshman admit rate for Asian applicants at UC Santa Cruz (population of interest) change, measured in percentage point difference (metric being measured)?

In [ ]:
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# PAGE SETUP
# ============================================================

st.set_page_config(
    page_title="UCSC Admissions Dashboard",
    layout="wide",
    initial_sidebar_state="collapsed"
)

# Reduce Streamlit padding so everything fits on one screen
st.markdown("""
<style>
    .block-container {
        padding-top: 1rem;
        padding-bottom: 0rem;
        padding-left: 2rem;
        padding-right: 2rem;
        max-width: 100%;
    }

    h1 {
        margin-top: 0rem;
        margin-bottom: 0rem;
        font-size: 2rem !important;
    }

    h3 {
        margin-top: 0.2rem;
        margin-bottom: 0.2rem;
    }

    [data-testid="stMetric"] {
        padding: 5px 10px;
    }

    [data-testid="stMetricValue"] {
        font-size: 1.7rem;
    }

    #MainMenu {
        visibility: hidden;
    }

    footer {
        visibility: hidden;
    }
</style>
""", unsafe_allow_html=True)


# ============================================================
# LOAD DATA
# ============================================================

DATA_FILE = (
    Path(__file__).parent /
    "uc_admissions_summary_by_ethnicity.csv"
)

@st.cache_data
def load_data():
    return pd.read_csv(DATA_FILE)

try:
    df = load_data()

except Exception as e:
    st.error("Could not load dataset.")
    st.exception(e)
    st.stop()


# ============================================================
# FILTER DATA
# ============================================================

ucsc_asian = df[
    (df["entrant_level"].astype(str).str.lower() == "freshman")
    & (df["campus"].astype(str).str.strip() == "Santa Cruz")
    & (df["ethnicity"].astype(str).str.strip() == "Asian")
    & (df["fall_term"].between(2021, 2025))
    & (df["count_type"].isin(["App", "Adm"]))
].copy()


# ============================================================
# CREATE SUMMARY
# ============================================================

summary = ucsc_asian.pivot_table(
    index="fall_term",
    columns="count_type",
    values="n",
    aggfunc="sum"
).reset_index()

summary["admit_rate"] = (
    summary["Adm"] / summary["App"]
) * 100

summary["admit_rate"] = summary["admit_rate"].round(2)


# ============================================================
# CALCULATIONS
# ============================================================

rate_2021 = summary.loc[
    summary["fall_term"] == 2021,
    "admit_rate"
].iloc[0]

rate_2025 = summary.loc[
    summary["fall_term"] == 2025,
    "admit_rate"
].iloc[0]

pp_change = rate_2025 - rate_2021


app_2021 = summary.loc[
    summary["fall_term"] == 2021,
    "App"
].iloc[0]

app_2025 = summary.loc[
    summary["fall_term"] == 2025,
    "App"
].iloc[0]


adm_2021 = summary.loc[
    summary["fall_term"] == 2021,
    "Adm"
].iloc[0]

adm_2025 = summary.loc[
    summary["fall_term"] == 2025,
    "Adm"
].iloc[0]


app_growth = (
    (app_2025 / app_2021) - 1
) * 100

admit_growth = (
    (adm_2025 / adm_2021) - 1
) * 100


# ============================================================
# DASHBOARD HEADER
# ============================================================

st.title("🎓 UC Santa Cruz Asian Freshman Admissions")

st.caption(
    "Fall 2021–Fall 2025 | "
    "How did the freshman admit rate for Asian applicants change?"
)


# ============================================================
# KPI ROW
# ============================================================

k1, k2, k3, k4, k5 = st.columns(5)

k1.metric(
    "2021 Admit Rate",
    f"{rate_2021:.2f}%"
)

k2.metric(
    "2025 Admit Rate",
    f"{rate_2025:.2f}%"
)

k3.metric(
    "Rate Change",
    f"+{pp_change:.2f} pp"
)

k4.metric(
    "Applicant Growth",
    f"+{app_growth:.1f}%"
)

k5.metric(
    "Admit Growth",
    f"+{admit_growth:.1f}%"
)


# ============================================================
# MAIN DASHBOARD ROW
# ============================================================

left, right = st.columns([1.25, 1])


# ============================================================
# LEFT CHART — ADMIT RATE
# ============================================================

with left:

    st.markdown("### Admit Rate Trend")

    fig1, ax1 = plt.subplots(figsize=(7, 3.1))

    ax1.plot(
        summary["fall_term"],
        summary["admit_rate"],
        marker="o",
        linewidth=2
    )

    ax1.set_ylabel("Admit Rate (%)")
    ax1.set_xlabel("Fall Term")

    ax1.set_xticks(summary["fall_term"])

    ax1.set_ylim(40, 90)

    ax1.grid(
        axis="y",
        alpha=0.25
    )

    for year, rate in zip(
        summary["fall_term"],
        summary["admit_rate"]
    ):
        ax1.annotate(
            f"{rate:.1f}%",
            (year, rate),
            textcoords="offset points",
            xytext=(0, 7),
            ha="center",
            fontsize=9
        )

    fig1.tight_layout()

    st.pyplot(
        fig1,
        use_container_width=True
    )

    plt.close(fig1)


# ============================================================
# RIGHT CHART — APPLICANTS VS ADMITS
# ============================================================

with right:

    st.markdown("### Applicants vs. Admits")

    fig2, ax2 = plt.subplots(figsize=(6, 3.1))

    x = range(len(summary))

    width = 0.35

    ax2.bar(
        [i - width / 2 for i in x],
        summary["App"],
        width,
        label="Applicants"
    )

    ax2.bar(
        [i + width / 2 for i in x],
        summary["Adm"],
        width,
        label="Admits"
    )

    ax2.set_xticks(list(x))

    ax2.set_xticklabels(
        summary["fall_term"]
    )

    ax2.set_xlabel("Fall Term")

    ax2.set_ylabel("Students")

    ax2.legend(
        frameon=False,
        fontsize=8
    )

    ax2.grid(
        axis="y",
        alpha=0.2
    )

    fig2.tight_layout()

    st.pyplot(
        fig2,
        use_container_width=True
    )

    plt.close(fig2)


# ============================================================
# BOTTOM SUMMARY
# ============================================================

st.info(
    f"📊 **Key Finding:** The Asian freshman admit rate at UC Santa Cruz "
    f"increased from **{rate_2021:.2f}% in Fall 2021** to "
    f"**{rate_2025:.2f}% in Fall 2025**, an increase of "
    f"**{pp_change:.2f} percentage points**. "
    f"During the same period, applicants increased "
    f"**{app_growth:.1f}%**, while admits increased "
    f"**{admit_growth:.1f}%**."
)